In [ ]:
%load_ext autoreload
%autoreload 2

from seek_code import SEEK
from data_handler import EleHandler
from classifier import Classifier
from backbone import Backbone
from concept_head import ConceptHead
from retrieval import Retrieval
from projector import Projector

from torch.utils.data import DataLoader, Subset
from torch.utils.data.sampler import RandomSampler

import torch, math
import torch.nn as nn
import torch.optim as optim
import pandas as pd

In [ ]:
training_dataset = EleHandler(subset='IDI_training') # 70% cut along elephants
print(len(training_dataset))
train_indices, val_indices, test_indices = training_dataset.split_along_encounters([0.7,0.15,0.15], hour_delta=24)
train_loader, val_loader, test_loader = (DataLoader(training_dataset, batch_size=128, sampler=RandomSampler(indices), num_workers=4) for indices in [train_indices, val_indices, test_indices])

assert not (set(train_indices) & set(val_indices)), "Train and Validation sets overlap!"
assert not (set(train_indices) & set(test_indices)), "Train and Test sets overlap!"
assert not (set(val_indices) & set(test_indices)), "Validation and Test sets overlap!"


# odd elephants are used as the query set
retrieval_dataset = EleHandler(subset='IDI_retrieval') # 30% cut along elephants
print(len(retrieval_dataset))

gallery_indices, query_indices, _ = retrieval_dataset.split_along_encounters([0.49,0.49,0.02], hour_delta=24)
gallery_loader, query_loader = (DataLoader(retrieval_dataset, batch_size=128, sampler=RandomSampler(indices), num_workers=4) for indices in [gallery_indices, query_indices])
assert not (set(gallery_indices) & set(query_indices)), "Gallery and Query sets overlap!"
assert not (set(train_indices) & set(gallery_indices)), "Train and Gallery sets overlap!"

# Check the values of the labels
num_training_classes = max(torch.max(batch[2]) for batch in train_loader) + 1
print('Number of elephants in training set:', num_training_classes)

num_query_classes = max(torch.max(batch[2]) for batch in query_loader) + 1
print('Number of (different) elephants in query set:', num_query_classes)

code = 'drafts'

In [ ]:
ba = Backbone(with_ears = True, pretraining = "savannah_elephants", experiment_code=code)
ch = ConceptHead(experiment_code= code)
ch.test(test_loader,ba)

cl = Classifier(num_classes=num_training_classes, experiment_code=code)
#cl.train(train_loader, val_loader, ba, num_epochs=30)
cl.test(test_loader,ba)

pr = Projector(experiment_code=code)
pr.train(train_loader, val_loader, ba, ch, cl, num_epochs=30)
pr.test(test_loader,ba, ch, cl)